# Kaggle Submission: exp_002 native checkpoint

This notebook is intended for direct upload into the BirdCLEF+ 2026 Kaggle competition.

What you need before running it:
- Attach the competition data.
- Attach a Kaggle Dataset that contains the `exp_002` checkpoint `best_model.pt`.
- Keep internet disabled.

What it does:
- Finds the competition data and the attached `exp_002` checkpoint.
- Reconstructs the repository-native waveform model used in `exp_002`.
- Runs direct 5-second soundscape inference with a single checkpoint.
- Saves `/kaggle/working/submission.csv`.

Important note:
- This notebook is intentionally a pure `exp_002` submission path.
- It does not use the external `LB862` / `LB872` blend.
- It does not apply the Perch-style metadata priors or the reference public-LB heuristics.


In [ ]:
import os
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")
START = time.time()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))

torch.set_float32_matmul_precision("high")
print("Torch:", torch.__version__)
print("Device:", device)
if device.type == "cpu":
    print("CPU threads:", torch.get_num_threads())


## Configuration

- `MODEL_DATASET_HINT` is optional. Set it only if you know the exact dataset folder name under `/kaggle/input/`.
- `CHECKPOINT_NAME` defaults to `best_model.pt`.
- The resolver searches recursively, so it can handle both a flat dataset and a dataset that preserves the local folder tree.


In [ ]:
MODEL_DATASET_HINT = None  # Example: "birdclef-2026-exp002-model"
CHECKPOINT_NAME = "best_model.pt"

@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)


def resolve_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026"),
    ]
    for candidate in candidates:
        if (candidate / "sample_submission.csv").exists():
            return candidate
    raise FileNotFoundError("BirdCLEF competition data was not found under /kaggle/input.")


def iter_candidate_roots(model_dataset_hint: str | None):
    input_root = Path("/kaggle/input")

    if model_dataset_hint:
        hinted_path = Path(model_dataset_hint)
        if hinted_path.exists():
            yield hinted_path
        else:
            hinted_under_input = input_root / model_dataset_hint
            if hinted_under_input.exists():
                yield hinted_under_input

    for child in sorted(input_root.iterdir()):
        yield child


def resolve_checkpoint_path(model_dataset_hint: str | None = None, checkpoint_name: str = "best_model.pt") -> Path:
    searched = []
    seen = set()

    for root in iter_candidate_roots(model_dataset_hint):
        try:
            root = root.resolve()
        except FileNotFoundError:
            continue

        root_key = str(root)
        if root_key in seen:
            continue
        seen.add(root_key)
        searched.append(root_key)

        if root.is_file() and root.name == checkpoint_name:
            return root

        if root.is_dir():
            matches = sorted(root.rglob(checkpoint_name))
            if matches:
                return matches[0]

    raise FileNotFoundError(
        f"Could not find {checkpoint_name} under /kaggle/input. Searched roots: {searched[:10]}"
    )


cfg = Config()
DATA_ROOT = resolve_data_root()
CHECKPOINT_PATH = resolve_checkpoint_path(MODEL_DATASET_HINT, CHECKPOINT_NAME)
SAMPLE_SUB = pd.read_csv(DATA_ROOT / "sample_submission.csv")
SPECIES = SAMPLE_SUB.columns[1:].tolist()
TEST_DIR = DATA_ROOT / "test_soundscapes"
TRAIN_SC_DIR = DATA_ROOT / "train_soundscapes"

print({
    "data_root": str(DATA_ROOT),
    "checkpoint_path": str(CHECKPOINT_PATH),
    "num_classes": len(SPECIES),
    "device": str(device),
})


## Model Blocks

The architecture below matches the repository-native `exp_002` training notebook.

Key detail:
- the checkpoint stores the full `WaveformSEDClassifier` state dict
- this includes the mel frontend inside the model wrapper
- inference therefore runs directly on waveform chunks


In [ ]:
class GEMFreqPool(nn.Module):
    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)
        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        clipwise_prob = torch.sigmoid(clipwise_logit)
        return {
            "clipwise_logit": clipwise_logit,
            "clipwise_prob": clipwise_prob,
        }


class SEDModel(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=False,
            in_chans=cfg.in_channels,
            features_only=False,
            global_pool="",
            num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(feat_dim, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.gem_pool(features)
        return self.head(pooled)


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.fmin,
            f_max=cfg.fmax,
            power=cfg.power,
            norm=cfg.norm,
            mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.mel(waveforms)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)
        batch_size = mel.shape[0]
        mel_flat = mel.reshape(batch_size, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


class WaveformSEDClassifier(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.mel = MelSpectrogramTransform(cfg)
        self.model = SEDModel(cfg)

    def forward(self, waveforms):
        mel = self.mel(waveforms)
        return self.model(mel)


## Load Checkpoint

This cell reconstructs the `exp_002` waveform model and loads the single best checkpoint.


In [ ]:
def safe_load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]
    return ckpt


def extract_checkpoint_meta(ckpt):
    if not isinstance(ckpt, dict):
        return {"stage": "raw_state_dict"}
    metrics = ckpt.get("metrics", {})
    return {
        "epoch": ckpt.get("epoch"),
        "metrics": metrics if isinstance(metrics, dict) else {},
        "config": ckpt.get("config", {}) if isinstance(ckpt.get("config", {}), dict) else {},
    }


def load_model(ckpt_path: Path):
    ckpt = safe_load_checkpoint(ckpt_path)
    model = WaveformSEDClassifier(cfg)
    model.load_state_dict(extract_state_dict(ckpt), strict=True)
    model.to(device).eval()
    return model, extract_checkpoint_meta(ckpt)


model, checkpoint_meta = load_model(CHECKPOINT_PATH)
print("Loaded checkpoint:", CHECKPOINT_PATH.name)
print(checkpoint_meta)


## Build Submission Row Mapping

The notebook uses `sample_submission.csv` to reconstruct the expected output order. When hidden test files are not mounted yet, it can still run in fallback mode for a quick dry check.


In [ ]:
row_pattern = re.compile(r"^(.*)_(\d+)$")


def parse_row_id(row_id: str):
    match = row_pattern.match(str(row_id))
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


expected_ids = SAMPLE_SUB["row_id"].tolist()
expected_by_stem = {}
for row_id in expected_ids:
    stem, end_sec = parse_row_id(row_id)
    if stem is None:
        continue
    expected_by_stem.setdefault(stem, []).append(end_sec)

for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(TEST_DIR.glob("*.ogg"))

if len(test_files) == 0:
    print("No hidden test audio is currently mounted. This is normal before Kaggle scoring starts.")
    fallback_files = []
    for stem in sorted(expected_by_stem):
        candidate = TRAIN_SC_DIR / f"{stem}.ogg"
        if candidate.exists():
            fallback_files.append(candidate)
    test_files = fallback_files
    print(f"Aligned fallback files found: {len(test_files)}")
else:
    print(f"Hidden test files found: {len(test_files)}")


## Inference And Submission

This is the pure `exp_002` path:
- one checkpoint
- no external blend
- no added metadata priors
- no reference public-LB postprocessing


In [ ]:
CHUNK = cfg.chunk_samples


def load_soundscape(path: Path) -> np.ndarray:
    audio, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def build_submission(row_ids, preds, expected_ids, species):
    if len(preds) == 0:
        pred_df = pd.DataFrame(
            np.zeros((0, len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index([], name="row_id"),
        )
    else:
        pred_df = pd.DataFrame(
            np.asarray(preds, dtype=np.float32),
            columns=species,
            index=pd.Index(row_ids, name="row_id"),
        )

    pred_df = pred_df[~pred_df.index.duplicated(keep="first")]

    missing_ids = [row_id for row_id in expected_ids if row_id not in pred_df.index]
    if missing_ids:
        zeros = pd.DataFrame(
            np.zeros((len(missing_ids), len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index(missing_ids, name="row_id"),
        )
        pred_df = pd.concat([pred_df, zeros], axis=0)

    pred_df = pred_df.loc[expected_ids]
    return pred_df.reset_index()


all_row_ids = []
all_preds = []

print("Running exp_002 native inference...")
with torch.no_grad():
    for file_idx, path in enumerate(test_files, start=1):
        stem = path.stem
        audio = load_soundscape(path)

        if stem in expected_by_stem:
            n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration)
        else:
            n_chunks = max(1, int(np.ceil(len(audio) / CHUNK)))

        padded_len = n_chunks * CHUNK
        if len(audio) < padded_len:
            audio = np.pad(audio, (0, padded_len - len(audio)))
        else:
            audio = audio[:padded_len]

        chunks = audio.reshape(n_chunks, CHUNK)
        chunk_tensor = torch.from_numpy(chunks).float().to(device)
        output = model(chunk_tensor)
        probs = torch.nan_to_num(output["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0)
        probs = torch.clamp(probs, 0.0, 1.0).detach().cpu().numpy().astype(np.float32)

        if stem in expected_by_stem:
            valid_indices = [
                (end_sec // int(cfg.chunk_duration)) - 1
                for end_sec in expected_by_stem[stem]
            ]
            valid_indices = [idx for idx in valid_indices if 0 <= idx < n_chunks]
            target_row_ids = [f"{stem}_{(idx + 1) * int(cfg.chunk_duration)}" for idx in valid_indices]
            selected_probs = probs[valid_indices]
        else:
            target_row_ids = [f"{stem}_{(idx + 1) * int(cfg.chunk_duration)}" for idx in range(n_chunks)]
            selected_probs = probs

        all_row_ids.extend(target_row_ids)
        all_preds.extend(selected_probs)

        if file_idx % 10 == 0 or file_idx == len(test_files):
            print(f"Processed {file_idx}/{len(test_files)} files")

submission = build_submission(all_row_ids, all_preds, expected_ids, SPECIES)
submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.to_csv("/kaggle/working/submission_exp_002_native.csv", index=False)

print("Saved: /kaggle/working/submission.csv")
print("Saved: /kaggle/working/submission_exp_002_native.csv")
print("Shape:", submission.shape)
print("Total runtime (sec):", round(time.time() - START, 1))
submission.head()


## What To Do On Kaggle

1. Upload this notebook to the BirdCLEF+ 2026 competition.
2. Attach the competition data.
3. Attach your checkpoint dataset containing `best_model.pt` from `exp_002`.
4. If your dataset preserves folder structure, that is fine; the notebook searches recursively.
5. Run `Save Version` and choose `Save & Submit`.
6. After Kaggle finishes scoring, copy the public LB score back into the project notes.
